# Camada Bronze

Este notebook realiza a persistência dos dados coletados na camada Bronze do pipeline.

A camada Bronze tem como objetivo preservar os dados provenientes das fontes de origem com o mínimo de transformação necessária para armazenamento estruturado.

Fontes utilizadas:

- Binance API — criptomoedas
- Banco Central do Brasil — Dólar/Real, Meta Selic e IPCA
- B3 — Ibovespa

As tabelas serão armazenadas no formato Delta no ambiente Databricks.

## Preparação do ambiente

Os notebooks de coleta são executados para disponibilizar os DataFrames necessários para persistência na camada Bronze.

In [0]:
%run ./01_coleta_criptomoedas

In [0]:
%run ./02_coleta_indicadores_macro

In [0]:
%run ./03_coleta_ibovespa

## Configuração do catálogo e schema

Os dados da camada Bronze serão armazenados em um schema específico denominado `bronze`.

O catálogo atual do ambiente Databricks será utilizado para criação das tabelas.

In [0]:
catalogo_atual = spark.sql(
    "SELECT current_catalog() AS catalogo"
).collect()[0]["catalogo"]

schema_atual = spark.sql(
    "SELECT current_schema() AS schema"
).collect()[0]["schema"]

print("Catálogo atual:", catalogo_atual)
print("Schema atual:", schema_atual)

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Camada Bronze do MVP de Engenharia de Dados'
""")

print("Schema bronze criado ou já existente.")

## Metadados de ingestão

Antes da persistência, são adicionadas informações de rastreabilidade aos dados, indicando a fonte e o momento de ingestão.

Esses metadados permitem identificar a origem dos registros e auxiliam no acompanhamento do pipeline.

In [0]:
from datetime import datetime

df_criptos_bronze = df_criptos.copy()

df_criptos_bronze["fonte"] = "Binance API"
df_criptos_bronze["data_ingestao"] = datetime.now()

display(df_criptos_bronze.head())

In [0]:
df_dolar_bronze = df_dolar.copy()

df_dolar_bronze["fonte"] = "Banco Central do Brasil - PTAX"
df_dolar_bronze["data_ingestao"] = datetime.now()

display(df_dolar_bronze.head())

In [0]:
df_selic_bronze = df_selic.copy()

df_selic_bronze["fonte"] = "Banco Central do Brasil - SGS 432"
df_selic_bronze["data_ingestao"] = datetime.now()

display(df_selic_bronze.head())

In [0]:
df_ipca_bronze = df_ipca.copy()

df_ipca_bronze["fonte"] = "Banco Central do Brasil - SGS 433"
df_ipca_bronze["data_ingestao"] = datetime.now()

display(df_ipca_bronze.head())

In [0]:
df_ibov_bronze = df_ibov.copy()

df_ibov_bronze["fonte"] = "B3 - BVBG.087.01 IndexReport"
df_ibov_bronze["data_ingestao"] = datetime.now()

display(df_ibov_bronze.head())

## Conversão para Spark DataFrames

Os dados coletados foram inicialmente manipulados em Pandas.

Nesta etapa, os DataFrames são convertidos para Spark DataFrames para utilização dos recursos nativos de armazenamento e processamento do Databricks.

In [0]:
spark_criptos = spark.createDataFrame(df_criptos_bronze)
spark_dolar = spark.createDataFrame(df_dolar_bronze)
spark_selic = spark.createDataFrame(df_selic_bronze)
spark_ipca = spark.createDataFrame(df_ipca_bronze)
spark_ibov = spark.createDataFrame(df_ibov_bronze)

print("DataFrames convertidos para Spark.")

## Persistência das tabelas Bronze

Os dados são armazenados como tabelas Delta gerenciadas pelo Databricks.

O modo `overwrite` permite que a execução do notebook seja reproduzível, substituindo a versão anterior das tabelas quando necessário.

In [0]:
spark_criptos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.criptomoedas")

spark_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.dolar")

spark_selic.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.selic")

spark_ipca.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.ipca")

spark_ibov.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.ibovespa")

print("Tabelas Bronze gravadas com sucesso.")

## Validação da camada Bronze

Após a persistência, são verificadas as tabelas criadas e a quantidade de registros armazenados em cada uma delas.

In [0]:
tabelas_bronze = [
    "criptomoedas",
    "dolar",
    "selic",
    "ipca",
    "ibovespa"
]

print("=== VALIDAÇÃO CAMADA BRONZE ===")

for tabela in tabelas_bronze:

    quantidade = spark.sql(
        f"SELECT COUNT(*) AS qtd FROM bronze.{tabela}"
    ).collect()[0]["qtd"]

    print(f"{tabela}: {quantidade} registros")

In [0]:
spark.sql("""
SHOW TABLES IN bronze
""").show(truncate=False)

## Resultado

Os dados provenientes das diferentes fontes foram persistidos com sucesso na camada Bronze utilizando tabelas Delta.

Nesta camada foram preservadas as informações coletadas nas fontes de origem, acrescidas apenas de metadados de rastreabilidade.

A próxima etapa do pipeline será a construção da camada Silver, responsável pela padronização, tratamento e integração dos dados.